# Protocols

> The contracts between models, objectives, solvers and policies, as in stable-worldmodel.

In [ ]:
#| default_exp protocols

In [ ]:
#| hide
from nbdev.showdoc import *

The same protocols, names and signatures as stable-worldmodel, so models and solvers written
for one library fit the other. Only the documented tensor shapes gain an **agent axis**:

| tensor | stable-worldmodel | stable-marl |
|---|---|---|
| action candidates | `(B, S, horizon, action_dim)` | `(B, S, horizon, num_agents)` discrete actions, or `(B, S, horizon, num_agents, action_dim)` |
| cost | `(B, S)` | `(B, S)` joint, or `(B, S, num_agents)` per agent |
| context frames | `pixels` `(B, S, H, C, h, w)` | per agent: `pixels` `(B, S, H, num_agents, C, h, w)` (each agent's view) |
| constraints | `(B, S, C)` | `(B, S, C)`, or `(B, S, num_agents, C)` per agent |

`B` is the number of envs, `S` the number of candidate plans (samples) per env, `H` the number
of context steps. A joint cost scores the plan of all agents together (centralised planning); a
per-agent cost lets a solver plan for each agent (decentralised planning).

The protocols are structural (``typing.Protocol``): a class satisfies one by having the
methods, without inheriting from it. torch is only needed by the classes implementing them.

In [ ]:
#| export
# Adapted from stable-worldmodel's `protocols.py` (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

from typing import TYPE_CHECKING, Protocol, runtime_checkable

import numpy as np

if TYPE_CHECKING:
    import torch

In [ ]:
#| export
@runtime_checkable
class Costable(Protocol):
    """
    The cost surface planning solvers optimise: every solver types against it. Satisfied by a
    world model composed with an :class:`Objective` (a shooting cost evaluator), and by models
    exposing ``get_cost`` natively.
    """
    def criterion(self, info_dict: dict, action_candidates: torch.Tensor) -> torch.Tensor:
        """
        Cost criterion of the action candidates ``(B, S, horizon, num_agents[, action_dim])``:
        ``(B, S)`` joint or ``(B, S, num_agents)`` per agent.
        """
        ...

    def get_cost(self, info_dict: dict, action_candidates: torch.Tensor) -> torch.Tensor:
        "Cost of the action candidates, shaped like :meth:`criterion`."
        ...


@runtime_checkable
class Constrainable(Protocol):
    """
    The optional constraint surface of a cost object, feature-detected by constrained solvers
    with ``isinstance(cost, Constrainable)``. A constraint ``g_i`` is satisfied when ``g_i <= 0``.
    """
    def get_constraints(self, info_dict: dict, action_candidates: torch.Tensor) -> torch.Tensor:
        "Constraint values of the candidates: ``(B, S, C)``, or ``(B, S, num_agents, C)`` per agent."
        ...


@runtime_checkable
class Dynamics(Protocol):
    "The dynamics surface a shooting cost evaluator needs from a world model."

    def encode(self, x: dict) -> dict:
        "Embed raw observations (e.g. ``pixels``, ``pixels``, ``state``); returns `x` with the embeddings (e.g. ``emb``)."
        ...

    def rollout(self, info_dict: dict, action_candidates: torch.Tensor) -> dict:
        """
        Roll the candidate action sequences ``(B, S, horizon, num_agents[, action_dim])`` forward.

        `info_dict` holds ``H`` context steps of observations (e.g. ``pixels`` of shape
        ``(B, S, H, num_agents, C, h, w)``); when ``H > 1`` the actions executed between them are
        given as ``action_history`` ``(B, S, H - 1, num_agents[, action_dim])``. Returns
        `info_dict` with the rollout outputs, e.g. ``predicted_emb`` of shape
        ``(B, S, H + horizon, ...)`` whose first ``H`` entries encode the context.
        """
        ...


@runtime_checkable
class Objective(Protocol):
    """
    Scores an *already rolled-out* `info_dict` (holding e.g. ``predicted_emb`` and the goal):
    unlike :class:`Costable`, it gets no action candidates and does no rollout. A shooting cost
    evaluator stores the candidates under ``action_candidates`` for action-space penalties.
    """
    def __call__(self, info_dict: dict) -> torch.Tensor:
        "Per-candidate cost: ``(B, S)`` joint or ``(B, S, num_agents)`` per agent."
        ...


@runtime_checkable
class Transformable(Protocol):
    "Reversible data transformation (normalizers, scalers), e.g. a ``process`` entry of a policy."

    def transform(self, x: np.ndarray) -> np.ndarray:
        ...

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        ...


@runtime_checkable
class Actionable(Protocol):
    "A model computing actions from an info dict, e.g. the model of a :class:`FeedForwardPolicy`."

    def get_action(self, info: dict, horizon: int = 1, prefix_actions: torch.Tensor | None = None) -> torch.Tensor:
        """
        Actions for the info dict: ``(..., num_agents[, action_dim])`` when ``horizon == 1``,
        else a sequence ``(..., horizon, num_agents[, action_dim])``. `prefix_actions`
        ``(..., t, num_agents[, action_dim])``, ``t < horizon``, are applied first (warm start).
        """
        ...

### Tests

In [ ]:
class Model:
    def get_action(self, info, horizon=1, prefix_actions=None): ...
    def encode(self, x): ...
    def rollout(self, info_dict, action_candidates): ...

class Scaler:
    def transform(self, x): return x
    def inverse_transform(self, x): return x

class Cost:
    def criterion(self, info_dict, action_candidates): ...
    def get_cost(self, info_dict, action_candidates): ...

assert isinstance(Model(), Actionable) and isinstance(Model(), Dynamics) and not isinstance(Model(), Costable)
assert isinstance(Scaler(), Transformable) and isinstance(Cost(), Costable) and not isinstance(Cost(), Constrainable)
assert isinstance(lambda info_dict: 0, Objective)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()